# 05 — The risk model Σ(t), and does it work?

> **Universe build** (`ryan/universe`). Same model code as `ryan/sample`; the
> universe, tables and history come from `config.py` — profile set by the
> `BARRA_UNIVERSE` environment variable (default `r1000`). See the README.

**Step 7.** Assemble

$$\Sigma(t) = X(t)\,F(t)\,X(t)^\top + \Delta(t)$$

and test it out of sample. **Σ is never materialized**, apart from one
verification cell. Every quantity goes through factor space:

```python
y = X.T @ w;  var = y @ F @ y + (w**2 * delta).sum()      # O(NK + K²), not O(N²K)
```

The machinery lives in `riskmodel.py` (`ModelStore`, `RiskModel`). 06–08 use
the same objects.

### What "works" means

Every test pairs a forecast made at *t* with the outcome at *t + 1*, which the
forecast never saw. The **bias statistic** is the standard deviation of
$z = r(t+1)/\hat\sigma(t)$: 1 means calibrated and above 1 means risk was
understated. Over ~180 months the 95% band is about 1 ± 0.10.

Handover decision #2 is the reason this is the test. A single full-sample F
scores ~1.0 in sample and 1.43 out of sample, and an optimizer lives entirely
in the out-of-sample world.

Reads the 03 and 04 tables plus `ryan_<profile>_exposures`. Writes
`ryan_<profile>_bias_tests`.

In [ ]:
import os
import sys
import time
from dataclasses import replace

sys.path.insert(0, os.getcwd())

# ryan/universe/config.py -- imported FIRST: it shadows the sample's config for
# every shared module and puts ryan/sample (where the model code lives) on the path.
import config
assert config.PROFILE != "sample", "wrong config on the path — run this notebook from ryan/universe"
print(config.describe())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import biastest
import factor_cov
import optimizer
import riskmath
import riskmodel

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

INK, ACCENT, MUTED, GREEN, AMBER = "#2a78d6", "#e34948", "#8a8a83", "#1baf7a", "#eda100"
STAGE_COLORS = {"cov_raw": MUTED, "cov_nw": AMBER, "cov_eigen": GREEN, "cov": INK}
STAGE_NAMES = {"cov_raw": "raw", "cov_nw": "nw", "cov_eigen": "eigen", "cov": "final"}


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


# Long-only min-variance at every date for two F stages. At universe scale this
# runs on the projected-gradient solver (~2 s per solve at N=4000), so roughly
# 10-15 minutes. Off = skip it.
RUN_LONG_ONLY_TEST = True

In [ ]:
engine = db.get_engine()
db.check_connection(engine)
store = riskmodel.ModelStore.from_db(engine)
R = store.realized_returns()          # raw monthly excess returns, eom x id (untrimmed)
dates = store.dates
nxt = dict(zip(dates[:-1], dates[1:]))
print(f"{len(dates)} forecast dates, {len(dates) - 1} testable "
      f"({dates[0].date()} → {dates[-2].date()}, outcomes through {dates[-1].date()})")

---
## Stage 1 — Σ at the latest date, verified against the materialized matrix

This is the only place Σ is ever formed. At 3,000–4,000 names it is 70–130 MB of
float64 and an eigendecomposition taking seconds to tens of seconds, which is
fine once as a check. Nothing downstream relies on it.

In [ ]:
model = store.at()
N, K = model.X.shape
sigma = model.X.to_numpy() @ model.F.to_numpy() @ model.X.to_numpy().T + np.diag(model.delta.to_numpy())
ev = np.linalg.eigvalsh(sigma)
w_test = pd.Series(np.random.default_rng(0).dirichlet(np.ones(N)), index=model.ids)
dense = 12 * w_test.to_numpy() @ sigma @ w_test.to_numpy()

checks = {
    "factor-space variance == wᵀΣw on the dense matrix": np.isclose(model.variance(w_test), dense),
    "Σ symmetric positive definite": ev.min() > 0,
    "smallest eigenvalue of Σ ≥ smallest specific variance (Δ keeps it off zero)":
        ev.min() >= model.delta.min() * (1 - 1e-9),
    "every priced name has an industry, a Δ and a ticker or name": bool(
        model.X.filter(like="ind_").sum(axis=1).eq(1).all() and model.delta.gt(0).all()
        and model.meta["conm"].notna().all()),
}
for k, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {k}")
print(f"\nN = {N} assets, K = {K} factors at {model.date.date()};  Σ condition number {ev.max() / ev.min():,.0f}")

t0 = time.perf_counter()
for _ in range(200):
    model.variance(w_test)
t_fac = (time.perf_counter() - t0) / 200
t0 = time.perf_counter()
for _ in range(200):
    w_test.to_numpy() @ sigma @ w_test.to_numpy()
t_dense = (time.perf_counter() - t0) / 200
print(f"per-call: factor space {t_fac * 1e6:.0f} µs, dense (Σ already built) {t_dense * 1e6:.0f} µs. "
      "The dense figure excludes the O(N²K) cost of building Σ at all.")

In [ ]:
vols = model.asset_vols()
mkt = model.market_weights()
dec = model.decompose(mkt)
print(f"cap-weighted sample market: predicted vol {dec['total_vol']:.1%}, "
      f"{dec['pct_factor']:.1%} of variance from factors")
print(f"single names: median total vol {vols['total_vol'].median():.1%}, "
      f"median share of variance that is specific {vols['pct_specific_var'].median():.0%}")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].hist(vols["total_vol"] * 100, bins=30, color=INK)
ax[0].axvline(dec["total_vol"] * 100, color=ACCENT, lw=2)
ax[0].annotate(" cap-weighted market", (dec["total_vol"] * 100, ax[0].get_ylim()[1] * .9), color=ACCENT, fontsize=9)
ax[0].set(title=f"Predicted total vol, {model.date.date()}", xlabel="%/yr", ylabel="assets")
ax[1].scatter(np.log10(model.meta["cap"]), vols["pct_specific_var"] * 100, s=14, color=INK, alpha=.7, lw=0)
ax[1].set(title="Share of variance that is specific, by size", xlabel="log10 market cap ($M)", ylabel="% specific")
for a in ax:
    tidy(a)
plt.tight_layout()
plt.show()

---
## Stage 2 — Single assets

$z_i = r_i(t+1) / \hat\sigma_i(t)$ with $\hat\sigma_i^2 = x_i^\top F x_i + \delta_i$,
using **raw** (unwinsorized) excess returns, because the realized outcome is
whatever happened. Single-stock returns are fat-tailed, and one +2225% month
moves a pooled std on its own. So there are three views: the pooled std, the
robust scale, and the median of the monthly cross-sectional bias statistics.

In [ ]:
rows = []
for d in dates[:-1]:
    m = store.at(d)
    r1 = R.loc[nxt[d]].reindex(m.ids)
    ok = r1.notna().to_numpy()
    v = m.asset_vols()["total_vol"] / np.sqrt(12)
    rows.append(pd.DataFrame({"date": d, "z": (r1 / v)[ok], "cap": m.meta["cap"][ok]}))
A = pd.concat(rows)

bt_eq = A.groupby("date")["z"].std()
bt_cw = A.groupby("date").apply(lambda g: np.sqrt(np.average(g["z"] ** 2, weights=g["cap"])), include_groups=False)
asset_stats = pd.Series({
    "forecasts": len(A),
    "pooled bias (std z)": A["z"].std(),
    "pooled, |z| ≤ 10 only": A.loc[A["z"].abs() <= 10, "z"].std(),
    "forecasts with |z| > 10 (count)": int((A["z"].abs() > 10).sum()),
    "robust scale (1.4826·MAD)": 1.4826 * np.median(np.abs(A["z"])),
    "median monthly B_t, equal-wtd": bt_eq.median(),
    "median monthly B_t, cap-wtd": bt_cw.median(),
    "MRAD, mean |B_t − 1| (eq-wtd)": riskmath.mrad(bt_eq.to_numpy()),
})
display(asset_stats.to_frame("value").round(3))

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(bt_eq.index, bt_eq, color=INK, lw=1.2, alpha=.5)
ax.plot(bt_eq.index, bt_eq.rolling(12).median(), color=INK, lw=2)
ax.annotate(" equal-wtd", (bt_eq.index[-1], bt_eq.rolling(12).median().iloc[-1]), color=INK, fontsize=9, va="center")
ax.plot(bt_cw.index, bt_cw.rolling(12).median(), color=ACCENT, lw=2)
ax.annotate(" cap-wtd", (bt_cw.index[-1], bt_cw.rolling(12).median().iloc[-1]), color=ACCENT, fontsize=9, va="center")
ax.axhline(1, color=MUTED, lw=1)
ax.set(title="Monthly cross-sectional bias statistic for single stocks (thin: monthly; thick: 12m rolling median)",
       ylabel="B_t", ylim=(0, 3))
ax.margins(x=.07)
tidy(ax)
plt.tight_layout()
plt.show()

The typical single-stock forecast is calibrated to slightly conservative
(median B_t below 1). The pooled std sits above 1 because a small fraction of
observations are enormous. That's what single-stock returns look like. A normal
forecast with σ right cannot be right about a +2225% month, and no variance
model is meant to be.

---
## Stage 3 — Portfolios, the test that matters for 06 and 07

At each date, build the portfolio from `Σ(t)` and score it on `t+1`:

- **cap-weighted** and **equal-weighted** sample portfolios: diversified and
  dominated by the market factor
- **random**: 50 fixed-seed long-only portfolios of 30 names, the "typical" portfolio
- **min-var (long-short)**: $\Sigma^{-1}\mathbf 1 / \mathbf 1^\top\Sigma^{-1}\mathbf 1$ via Woodbury,
  with no bounds, so it is the most aggressive consumer of Σ possible
- **min-var (long-only, ≤5%)**: what 07 actually does

Each is run under every stored F stage, with Δ held at its final stage. The
universe at *t* is the names that have a return at *t + 1*. A security that
delists during the month has no return in the data. That is a mild selection
effect, applied identically to every stage.

In [ ]:
rng = np.random.default_rng(20260927)
n_rand, k_rand = 50, 30


def ptf_z(w, m, r1):
    return float(w @ r1.reindex(w.index)) / (m.vol(w) / np.sqrt(12))


port = {s: {"cap-weighted": [], "equal-weighted": [], "random (median)": [], "min-var long-short": []}
        for s in STAGE_NAMES}
seeds = {d: int(rng.integers(1e9)) for d in dates[:-1]}
t0 = time.time()
for d in dates[:-1]:
    r1 = R.loc[nxt[d]]
    for s in STAGE_NAMES:
        m = store.at(d, s)
        m = m.subset(m.ids[r1.reindex(m.ids).notna().to_numpy()])
        port[s]["cap-weighted"].append(ptf_z(m.market_weights(), m, r1))
        port[s]["equal-weighted"].append(ptf_z(pd.Series(1 / len(m.ids), m.ids), m, r1))
        port[s]["min-var long-short"].append(ptf_z(optimizer.min_variance_closed_form(m), m, r1))
        rr = np.random.default_rng(seeds[d])
        zs = []
        for _ in range(n_rand):
            pick = rr.choice(m.ids, k_rand, replace=False)
            zs.append(ptf_z(pd.Series(rr.dirichlet(np.ones(k_rand)), pick), m, r1))
        port[s]["random (median)"].append(zs)
print(f"portfolio tests: {time.time() - t0:.0f}s")

In [ ]:
if RUN_LONG_ONLY_TEST:
    t0 = time.time()
    for s in ("cov_raw", "cov"):
        port[s]["min-var long-only ≤5%"] = []
        for d in dates[:-1]:
            r1 = R.loc[nxt[d]]
            m = store.at(d, s)
            m = m.subset(m.ids[r1.reindex(m.ids).notna().to_numpy()])
            w = optimizer.optimize(m, m.ids, max_weight=0.05)
            port[s]["min-var long-only ≤5%"].append(ptf_z(w, m, r1))
    print(f"long-only min-var: {time.time() - t0:.0f}s")


def stat(v):
    v = np.asarray(v)
    if v.ndim == 2:          # random portfolios: bias per portfolio, then the median
        return float(np.median(v.std(axis=0, ddof=1)))
    return riskmath.bias_stat(v)


ptf_table = pd.DataFrame({STAGE_NAMES[s]: {k: stat(v) for k, v in tests.items()} for s, tests in port.items()})
display(ptf_table.round(3))
lo_ci, hi_ci = riskmath.bias_ci(len(dates) - 1)
print(f"95% band: [{lo_ci:.3f}, {hi_ci:.3f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
tests = list(ptf_table.index)
xpos = np.arange(len(tests))
width = .2
ax.axhspan(lo_ci, hi_ci, color=MUTED, alpha=.15, lw=0)
for j, s in enumerate(STAGE_NAMES.values()):
    vals = ptf_table[s].to_numpy()
    col = list(STAGE_COLORS.values())[j]
    ax.bar(xpos + (j - 1.5) * width, vals, width * .92, color=col, label=s)
ax.axhline(1, color="#1a1a19", lw=1)
ax.set_xticks(xpos, tests)
ax.set(ylabel="bias statistic", title="Out-of-sample portfolio bias by F stage (shaded: 95% band)")
ax.legend(frameon=False, ncol=4, fontsize=9)
tidy(ax)
plt.tight_layout()
plt.show()

**Reading it.** On the sample, diversified portfolios were calibrated at every
stage, and the min-variance portfolios separated them: raw F 1.18 against final
1.07 long-short, 1.07 against 1.04 long-only. Look for the same pattern here.
With far more names the optimizer has more room to find F's weak directions,
so the gap between raw and final should be at least as wide.

---
## Stage 4 — Specific risk stages, at the portfolio level

F held at its final stage; Δ swapped between raw, shrunk and final.

In [ ]:
spec = store.specific
dz = {c: [] for c in ("specific_var_raw", "specific_var_shrunk", "specific_var")}
for d in dates[:-1]:
    r1 = R.loc[nxt[d]]
    m = store.at(d)
    m = m.subset(m.ids[r1.reindex(m.ids).notna().to_numpy()])
    sd = spec[spec["forecast_date"] == d].set_index("id")
    for c in dz:
        # Raw is NaN for the ~0.6% of rows on the size prior; those keep the prior.
        dv = sd[c].reindex(m.ids).fillna(sd["specific_var_shrunk"].reindex(m.ids))
        mm = replace(m, delta=dv)
        dz[c].append(ptf_z(optimizer.min_variance_closed_form(mm), mm, r1))
labels = {"specific_var_raw": "Δ raw", "specific_var_shrunk": "Δ shrunk", "specific_var": "Δ final"}
spec_table = pd.Series({labels[c]: riskmath.bias_stat(v) for c, v in dz.items()},
                       name="min-var long-short bias")
display(spec_table.round(3).to_frame())
print("On the sample Δ's stages barely moved the min-variance portfolio (1.064–1.068): its")
print("risk sits in factor directions it cannot diversify away. With thousands of names a")
print("min-var book can hold many low-specific-risk names, so Δ matters more here — the")
print("spread between the rows above is the measurement.")

---
## Stage 5 — Rolling vs single F (handover decision #2, measured)

`window=None` builds one F from all months and uses it at every date.
Scored on the same months it was fitted to, it looks perfect. The rolling F is
honest about what a live model knows.

In [ ]:
fl = db.read_table(config.TBL_FACTOR_RETURNS, engine)
fl["eom"] = pd.to_datetime(fl["eom"])
FR = fl.pivot(index="eom", columns="factor", values="factor_return")
FR = FR[factor_cov.canonical_order(FR.columns)]

full = factor_cov.estimate(FR, window=None, eigen_sims=0, verbose=False)
roll = factor_cov.estimate(FR, eigen_sims=0, vra_min_obs=10**6, verbose=False)
cmp_ = pd.DataFrame({
    "single full-sample F (in-sample)": biastest.summarize(full, FR, ["raw"]).iloc[0],
    "rolling 60m F, raw (out-of-sample)": biastest.summarize(roll, FR, ["raw"]).iloc[0],
    "rolling 60m F, final (out-of-sample)": biastest.summarize(
        factor_cov.path_from_long(db.read_table(config.TBL_FACTOR_COV, engine)), FR, ["final"]).iloc[0],
}).T
display(cmp_.round(3))
print("The full-sample column is the number that would have said 'ship it'. It cannot be")
print("reproduced live, because a live model never has next year's returns in its window.")

---
## Stage 6 — Write the summary

In [ ]:
out = []
for s, tests_ in port.items():
    for k, v in tests_.items():
        out.append({"level": "portfolio", "test": k, "stage": STAGE_NAMES[s], "bias": stat(v),
                    "n": len(v)})
for c, v in dz.items():
    out.append({"level": "portfolio", "test": "min-var long-short (Δ stage)", "stage": c,
                "bias": riskmath.bias_stat(v), "n": len(v)})
for k, v in asset_stats.items():
    out.append({"level": "asset", "test": k, "stage": "final", "bias": float(v), "n": len(A)})
for idx, row in cmp_.iterrows():
    for k, v in row.items():
        out.append({"level": "factor", "test": k, "stage": idx, "bias": float(v), "n": len(dates) - 1})
bias_tbl = pd.DataFrame(out)
db.write_table(bias_tbl, config.TBL_BIAS_TESTS, engine, unique_on=["level", "test", "stage"])
display(bias_tbl.head(12))

---
## Using the model

```python
store = riskmodel.ModelStore.from_db(engine)
model = store.at()                   # latest date; store.at("2020-03-31") for any other
model.vol(weights)                   # weights: pd.Series indexed by id
model.decompose(weights)             # factor / asset risk attribution
model.cov_row(asset_id)              # one row of Σ
```

- **06_portfolio_risk**: risk of a CSV portfolio
- **07_mean_variance**: optimize a CSV portfolio
- **08_asset_query**: everything the model says about one stock